In [10]:
import pandas as pd
import numpy as np
from calendar import monthrange
from sqlalchemy import create_engine, text
from pandas.tseries.offsets import MonthEnd

In [11]:
# def create_date_column(df):
#     # 각 월의 마지막 날짜 계산 함수
#     def get_last_day_of_month(year, month):
#         return monthrange(int(year), int(month))[1]
#
#     # 날짜 열 생성
#     df['Date'] = df.apply(
#         lambda row: pd.to_datetime(
#             f"{int(row['회계년'])}-{int(row['결산월'])}-{get_last_day_of_month(row['회계년'], row['결산월'])}"
#         ),
#         axis=1
#     )
#
#     # 주기가 분기형이면, 날짜를 분기 말일로 보정 (선택적 처리)
#     if '주기' in df.columns:
#         quarterly_mask = df['주기'].astype(str).str.contains("Q")
#         for idx in df[quarterly_mask].index:
#             base_date = df.loc[idx, 'Date']
#             df.loc[idx, 'Date'] = pd.date_range(end=base_date, periods=4, freq='3M')[-1]
#
#     return df

def convert_to_long_format(df):
    # 제거할 열
    drop_cols = ["결산월", "회계년", "주기"]

    # ID 변수 (고정값 유지할 열들)
    id_vars = ["Symbol", "company_name", "Date"]

    # 나머지는 전부 indicator 대상 열로 melt 처리
    value_vars = [col for col in df.columns if col not in id_vars + drop_cols]

    # melt 실행
    df_long = pd.melt(df,
                      id_vars=id_vars,
                      value_vars=value_vars,
                      var_name="indicator",
                      value_name="value")

    return df_long


def create_date_column(df):
    # 각 월의 마지막 날짜 계산 함수
    def get_last_day_of_month(year, month):
        return monthrange(int(year), int(month))[1]

    # 날짜 열 생성
    df['Date'] = df.apply(
        lambda row: pd.to_datetime(
            f"{int(row['회계년'])}-{int(row['결산월'])}-{get_last_day_of_month(row['회계년'], row['결산월'])}"
        ),
        axis=1
    )

    # 주기가 분기형이면, 날짜를 해당 분기 말일로 직접 지정
    if '주기' in df.columns:
        quarter_map = {
            '1Q': '-03-31',
            '2Q': '-06-30',
            '3Q': '-09-30',
            '4Q': '-12-31'
        }

        def override_to_quarter_end(row):
            q = str(row['주기']).strip()
            y = int(row['회계년'])
            return pd.to_datetime(f"{y}{quarter_map[q]}") if q in quarter_map else row['Date']

        df['Date'] = df.apply(override_to_quarter_end, axis=1)

    return df

def upload_fs_data_to_db(df_long, db_info, table_name="korea_fs_data", chunk_size=1000):
    """
    DB에 long format 재무 데이터를 테이블 생성 후 업로드함.
    기존 테이블은 없다고 가정하고 새로 생성함.
    """
    # ✅ 날짜 및 결측치 처리
    df_long['date'] = pd.to_datetime(df_long['date'])
    df_long = df_long.replace([np.inf, -np.inf], np.nan)
    df_long = df_long.where(pd.notnull(df_long), None)

    # ✅ DB 연결
    engine = create_engine(
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}"
    )
    conn = engine.raw_connection()
    cursor = conn.cursor()

    # ✅ 테이블 생성 쿼리
    create_table_sql = f"""
    CREATE TABLE IF NOT EXISTS `{table_name}` (
        `symbol` VARCHAR(20),
        `company_name` VARCHAR(50),
        `date` DATE,
        `indicator` VARCHAR(100),
        `value` DOUBLE,
        PRIMARY KEY (`symbol`, `date`, `indicator`)
    );
    """
    cursor.execute(create_table_sql)
    conn.commit()

    # ✅ 데이터 INSERT 쿼리
    insert_sql = f"""
    INSERT INTO `{table_name}` (`symbol`, `company_name`, `date`, `indicator`, `value`)
    VALUES (%s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        `company_name` = VALUES(`company_name`),
        `value` = VALUES(`value`);
    """

    # ✅ 튜플 리스트로 변환
    rows = df_long[["symbol", "company_name", "date", "indicator", "value"]].values.tolist()

    # ✅ Chunk 단위로 업로드
    for i in range(0, len(rows), chunk_size):
        chunk = rows[i:i+chunk_size]
        cursor.executemany(insert_sql, chunk)
        conn.commit()

    cursor.close()
    conn.close()
    print(f"✅ 총 {len(df_long)}개 row 업로드 완료 (중복은 자동 업데이트됨)")


    def convert_to_long_format(df):
        # 제거할 열
        drop_cols = ["결산월", "회계년", "주기"]

        # ID 변수 (고정값 유지할 열들)
        id_vars = ["Symbol", "company_name", "Date"]

        # 나머지는 전부 indicator 대상 열로 melt 처리
        value_vars = [col for col in df.columns if col not in id_vars + drop_cols]

        # melt 실행
        df_long = pd.melt(df,
                          id_vars=id_vars,
                          value_vars=value_vars,
                          var_name="indicator",
                          value_name="value")

        return df_long

In [15]:
path = r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA\is_bs_cf_Dataguide_2025_2Q.xlsx"
# sheet_name = "KOSPI"

raw_df = pd.read_excel(path)

# 기준 행
row_header_1 = 8  # 항목명
row_header_2 = 9  # 단위, 코드, 분류 등

# 복합 컬럼명 생성
def combine_headers(col):
    item = str(raw_df.loc[row_header_1, col]) if pd.notna(raw_df.loc[row_header_1, col]) else ""
    unit = str(raw_df.loc[row_header_2, col]) if pd.notna(raw_df.loc[row_header_2, col]) else ""
    combined = f"{unit.strip()}: {item.strip()}" if unit and item else item or unit
    return combined if combined else col  # fallback

# 새로운 컬럼 리스트 생성
new_columns = [combine_headers(col) for col in raw_df.columns]

# 컬럼명 적용
df_cleaned = raw_df.copy()
df_cleaned.columns = new_columns
df_cleaned = df_cleaned.iloc[10:].reset_index(drop=True)

# "Name"을 "company_name"으로 변경
df_cleaned = df_cleaned.rename(columns={"Name": "company_name"})

# 확인
print(df_cleaned.columns.tolist())

['Symbol', 'company_name', '결산월', '회계년', '주기', '매출액(천원)', '매출총이익(천원)', '영업이익(천원)', '계속사업이익(천원)', '당기순이익(천원)', '지배주주총포괄이익(천원)', '유형자산감가상각비(천원)', '연구개발비(천원)', '총자산(천원)', '유동자산(천원)', '당좌자산(천원)', '매출채권및기타채권(천원)', '재고자산(천원)', '투자부동산(천원)', '비유동부채(천원)', '총자본(천원)', '지배주주지분(천원)', '영업활동으로인한현금흐름(천원)', '영업활동으로인한현금흐름(직전4분기)(천원)', '영업활동으로인한현금흐름(평균)(천원)', '배당금지급(영업,투자,재무)(천원)']


In [17]:
# ======================
# 1. 주요 지표 컬럼 이름 찾기
# ======================
def find_column(df, keyword):
    for col in df.columns:
        if keyword in col and '(천원)' in col:
            return col
    return None

# 주요 항목들 컬럼명 식별
col_sales = find_column(df_cleaned, '매출액')
col_gross = find_column(df_cleaned, '매출총이익')
col_operating = find_column(df_cleaned, '영업이익')
col_continuing = find_column(df_cleaned, '계속사업이익')
col_net_income = find_column(df_cleaned, '당기순이익')
col_noncurrent_liab = find_column(df_cleaned, '비유동부채')

# 자본 컬럼 찾기
col_equity = None
for col in df_cleaned.columns:
    if "지배" in col and "지배" in col and "(천원)" in col:
        col_equity = col
        break

# 종목 코드 컬럼 찾기
col_symbol = None
for col in df_cleaned.columns:
    if col in ['Symbol', 'symbol', '종목코드', 'ticker', 'Ticker']:
        col_symbol = col
        break

# 회계년도 컬럼 찾기
col_year = None
for col in df_cleaned.columns:
    if col in ['회계년', '회계년도', 'year', 'Year']:
        col_year = col
        break

# 주기 컬럼 찾기
col_period = None
for col in df_cleaned.columns:
    if col in ['주기', '분기', 'quarter', 'Quarter', 'period']:
        col_period = col
        break

# 컬럼 찾기 결과 확인
print("=== 컬럼 매칭 결과 ===")
print(f"종목코드: {col_symbol}")
print(f"회계년: {col_year}")
print(f"주기: {col_period}")
print(f"매출액: {col_sales}")
print(f"매출총이익: {col_gross}")
print(f"영업이익: {col_operating}")
print(f"계속사업이익: {col_continuing}")
print(f"당기순이익: {col_net_income}")
print(f"비유동부채: {col_noncurrent_liab}")
print(f"자본: {col_equity}")
print()

# None인 컬럼 확인
missing_cols = []
col_dict = {
    '종목코드': col_symbol,
    '회계년': col_year,
    '주기': col_period,
    '매출액': col_sales,
    '매출총이익': col_gross,
    '영업이익': col_operating,
    '계속사업이익': col_continuing,
    '당기순이익': col_net_income,
    '비유동부채': col_noncurrent_liab,
    '자본': col_equity
}

for name, col in col_dict.items():
    if col is None:
        missing_cols.append(name)

if missing_cols:
    print(f"⚠️ 다음 컬럼을 찾지 못했습니다: {', '.join(missing_cols)}")
    print("\n사용 가능한 컬럼 목록:")
    for col in df_cleaned.columns:
        print(f"  - {col}")
    raise ValueError(f"필수 컬럼을 찾을 수 없습니다: {missing_cols}")

# ======================
# 2. 숫자형으로 변환
# ======================
for col in [col_sales, col_gross, col_operating, col_continuing, col_net_income, col_noncurrent_liab, col_equity]:
    df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# ======================
# 3. sort와 groupby 이용하여 YoY 증가율 계산
# ======================
df_cleaned = df_cleaned.sort_values(by=[col_symbol, col_year, col_period])

# YoY 증가율 계산 (4분기 전과 비교)
df_cleaned["YoY_매출액"] = df_cleaned.groupby(col_symbol)[col_sales].pct_change(periods=4, fill_method=None)
df_cleaned["YoY_매출총이익"] = df_cleaned.groupby(col_symbol)[col_gross].pct_change(periods=4, fill_method=None)
df_cleaned["YoY_영업이익"] = df_cleaned.groupby(col_symbol)[col_operating].pct_change(periods=4, fill_method=None)
df_cleaned["YoY_계속사업이익"] = df_cleaned.groupby(col_symbol)[col_continuing].pct_change(periods=4, fill_method=None)
df_cleaned["YoY_당기순이익"] = df_cleaned.groupby(col_symbol)[col_net_income].pct_change(periods=4, fill_method=None)

# ======================
# 4. 수익성 비율 계산
# ======================
# 수익성 비율
df_cleaned["매출총이익률"] = df_cleaned[col_gross] / df_cleaned[col_sales]
df_cleaned["영업이익률"] = df_cleaned[col_operating] / df_cleaned[col_sales]
df_cleaned["순이익률"] = df_cleaned[col_net_income] / df_cleaned[col_sales]

# 재무 안정성 비율
df_cleaned["비유동부채비율"] = df_cleaned[col_noncurrent_liab] / df_cleaned[col_equity]

# ======================
# 결과 미리보기
# ======================
display_cols = [
    col_symbol, col_year, col_period, col_sales,
    "YoY_매출액", "YoY_영업이익", "영업이익률", "순이익률", "비유동부채비율"
]
print("\n=== 결과 미리보기 ===")
print(df_cleaned[display_cols].dropna().head(10))

=== 컬럼 매칭 결과 ===
종목코드: Symbol
회계년: 회계년
주기: 주기
매출액: 매출액(천원)
매출총이익: 매출총이익(천원)
영업이익: 영업이익(천원)
계속사업이익: 계속사업이익(천원)
당기순이익: 당기순이익(천원)
비유동부채: 비유동부채(천원)
자본: 지배주주총포괄이익(천원)


=== 결과 미리보기 ===
        Symbol   회계년  주기     매출액(천원)   YoY_매출액  YoY_영업이익     영업이익률      순이익률  \
84924  A000020  2005  1Q  37319955.0  0.146695  0.382450  0.072820  0.022260   
84925  A000020  2005  2Q  36756509.0  0.111915 -0.046122  0.075784  0.039708   
84926  A000020  2005  3Q  38018668.0  0.067371 -0.073041  0.103189  0.046929   
84927  A000020  2005  4Q  40726455.0  0.108496  0.123421  0.152634  0.088089   
84928  A000020  2006  1Q  31652265.0 -0.151868 -0.396715  0.051797  0.018072   
84929  A000020  2006  2Q  34762702.0 -0.054244 -0.032748  0.077506  0.039307   
84930  A000020  2006  3Q  34930492.0 -0.081228 -0.489173  0.057372  0.029781   
84931  A000020  2006  4Q  47393164.0  0.163695  0.260151  0.165286  0.107325   
84932  A000020  2007  1Q  39592860.0  0.250870  2.737132  0.154751  0.089982   
84933  A000020  2007

In [18]:
df_cleaned = df_cleaned.dropna(subset=['회계년', '결산월'])

In [19]:
df_cleaned = create_date_column(df_cleaned)

In [20]:
df_long = convert_to_long_format(df_cleaned)

# inf → NaN, 그 뒤 객체형 변환 보정
df_long = df_long.replace([np.inf, -np.inf], np.nan).infer_objects(copy=False)

# 또는 특정 열만 안전하게 처리
df_long["value"] = pd.to_numeric(df_long["value"], errors="coerce")
df_long = df_long.dropna(subset=["value"])

new_col = ['symbol', 'company_name', 'date', 'indicator', 'value']
df_long.columns = new_col

fs_value_df = df_long

C:\Users\82108\AppData\Local\Temp\ipykernel_12744\4148504434.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_long = df_long.replace([np.inf, -np.inf], np.nan).infer_objects(copy=False)


In [21]:
df_long

,symbol,company_name,date,indicator,value
0,A000010,조흥은행,2004-03-31,매출액(천원),7.920890e+08
1,A000010,조흥은행,2004-06-30,매출액(천원),7.617880e+08
2,A000010,조흥은행,2004-09-30,매출액(천원),7.392400e+08
3,A000010,조흥은행,2004-12-31,매출액(천원),7.551090e+08
4,A000010,조흥은행,2005-03-31,매출액(천원),7.067880e+08
...,...,...,...,...,...
10054913,A950220,네오이뮨텍,2024-06-30,비유동부채비율,-1.512985e-01
10054914,A950220,네오이뮨텍,2024-09-30,비유동부채비율,-1.516142e+00
10054915,A950220,네오이뮨텍,2024-12-31,비유동부채비율,-3.173542e-01
10054916,A950220,네오이뮨텍,2025-03-31,비유동부채비율,-3.091104e-01


In [28]:
import pandas as pd
import numpy as np
from pandas.tseries.offsets import MonthEnd

# 1) 파일 로드
path = r"/DATA/DataGuide_Ratio_2025_2Q.xlsx"
raw_df = pd.read_excel(path, header=None)

# 2) 헤더 행 자동 탐지
header_row = None
for r in range(0, 30):
    row_vals = raw_df.iloc[r, :5].astype(str).str.strip().tolist()
    if row_vals[:5] == ['Symbol', 'Name', '결산월', '회계년', '주기']:
        header_row = r
        break
if header_row is None:
    header_row = 10  # fallback

# 3) 지표 코드/이름 행
indicator_code_row = header_row - 2
indicator_name_row = header_row - 1

# 4) 기본 칼럼 위치
symbol_col, name_col, mon_col, year_col, period_col = 0, 1, 2, 3, 4
first_indicator_col = 5
last_col = raw_df.shape[1] - 1
indicator_cols = list(range(first_indicator_col, last_col + 1))

# 5) 지표 이름·코드
indicator_names = raw_df.iloc[indicator_name_row, first_indicator_col:last_col+1].astype(str).str.strip().tolist()
indicator_codes = raw_df.iloc[indicator_code_row, first_indicator_col:last_col+1].astype(str).str.strip().tolist()

# 6) 데이터 블록
data_block = raw_df.iloc[header_row + 1:, :].copy()
is_repeat_header = data_block.iloc[:, 0].astype(str).str.strip().eq('Symbol')
data_block = data_block.loc[~is_repeat_header]

# 7) 열 이름 정리
data_block = data_block.rename(columns={
    symbol_col: 'symbol',
    name_col: 'company_name',
    mon_col: 'month',
    year_col: 'year'
})
data_block['year'] = pd.to_numeric(data_block['year'], errors='coerce')
data_block['month'] = pd.to_numeric(data_block['month'], errors='coerce')

# 8) 날짜 생성
valid_time = data_block['year'].notna() & data_block['month'].notna()
data_block.loc[valid_time, 'date'] = pd.to_datetime(dict(
    year=data_block.loc[valid_time, 'year'].astype(int),
    month=data_block.loc[valid_time, 'month'].astype(int),
    day=1
)) + MonthEnd(0)

# 9) 지표 칼럼 이름 정리
renames = {}
safe_indicator_names = []
for i, c in enumerate(indicator_cols):
    nm = indicator_names[i]
    if str(nm).lower() == 'nan' or nm.strip() == '':
        nm = indicator_codes[i]
    renames[c] = nm
    safe_indicator_names.append(nm)

data_block = data_block.rename(columns=renames)

# 10) long format 변환
id_cols = ['date', 'symbol', 'company_name']
value_cols = [c for c in safe_indicator_names if c in data_block.columns]
tidy_base = data_block[id_cols + value_cols].copy()

long_df = tidy_base.melt(
    id_vars=id_cols,
    value_vars=value_cols,
    var_name='indicator',
    value_name='value'
)

# 11) 값 숫자화 + 결측 제거
long_df['value'] = pd.to_numeric(long_df['value'], errors='coerce')
value_ratio_df = long_df.dropna(subset=['date','symbol','value']).reset_index(drop=True)

print(value_ratio_df.head(10))



        date   symbol company_name      indicator  value
0 2009-03-31  A005930         삼성전자  수정PER(연율화)(배)  40.95
1 2009-06-30  A005930         삼성전자  수정PER(연율화)(배)  11.07
2 2009-09-30  A005930         삼성전자  수정PER(연율화)(배)   9.31
3 2009-12-31  A005930         삼성전자  수정PER(연율화)(배)  11.39
4 2010-03-31  A005930         삼성전자  수정PER(연율화)(배)   8.66
5 2010-06-30  A005930         삼성전자  수정PER(연율화)(배)   7.89
6 2010-09-30  A005930         삼성전자  수정PER(연율화)(배)   7.67
7 2010-12-31  A005930         삼성전자  수정PER(연율화)(배)  12.23
8 2011-03-31  A005930         삼성전자  수정PER(연율화)(배)  14.60
9 2011-06-30  A005930         삼성전자  수정PER(연율화)(배)  10.16


In [34]:
# 삼성전자(A005930)의 수정PER(연율화)(배)만 추출
tested = value_ratio_df[
    (value_ratio_df['symbol'] == 'A005930') &
    (value_ratio_df['indicator'] == '수정PSR(연율화)(배)')
]
print(tested.tail())


             date   symbol company_name      indicator  value
165699 2024-06-30  A005930         삼성전자  수정PSR(연율화)(배)   1.87
165700 2024-09-30  A005930         삼성전자  수정PSR(연율화)(배)   1.32
165701 2024-12-31  A005930         삼성전자  수정PSR(연율화)(배)   1.19
165702 2025-03-31  A005930         삼성전자  수정PSR(연율화)(배)   1.24
165703 2025-06-30  A005930         삼성전자  수정PSR(연율화)(배)   1.35


In [30]:
concated_df = pd.concat([fs_value_df, value_ratio_df])

In [31]:
db_info = {
    "user": 'stox7412',
    "password": 'Apt106503!~',
    "host": '192.168.0.230',
    # 'host': 'hystox74.synology.me',
    "port": 3307,
    "database": "investar"
}

upload_fs_data_to_db(concated_df, db_info)

✅ 총 5527397개 row 업로드 완료 (중복은 자동 업데이트됨)


In [24]:
fs_value_df

,symbol,company_name,date,indicator,value
0,A000010,조흥은행,2004-03-31,매출액(천원),7.920890e+08
1,A000010,조흥은행,2004-06-30,매출액(천원),7.617880e+08
2,A000010,조흥은행,2004-09-30,매출액(천원),7.392400e+08
3,A000010,조흥은행,2004-12-31,매출액(천원),7.551090e+08
4,A000010,조흥은행,2005-03-31,매출액(천원),7.067880e+08
...,...,...,...,...,...
10054913,A950220,네오이뮨텍,2024-06-30,비유동부채비율,2.636177e-02
10054914,A950220,네오이뮨텍,2024-09-30,비유동부채비율,3.425031e-01
10054915,A950220,네오이뮨텍,2024-12-31,비유동부채비율,7.150773e-02
10054916,A950220,네오이뮨텍,2025-03-31,비유동부채비율,7.865029e-02


In [35]:
tested = concated_df[
    (concated_df['symbol'] == 'A005930') &
    (concated_df['indicator'] == '수정PSR(연율화)(배)')
]
print(tested.tail())

         symbol company_name       date      indicator  value
165699  A005930         삼성전자 2024-06-30  수정PSR(연율화)(배)   1.87
165700  A005930         삼성전자 2024-09-30  수정PSR(연율화)(배)   1.32
165701  A005930         삼성전자 2024-12-31  수정PSR(연율화)(배)   1.19
165702  A005930         삼성전자 2025-03-31  수정PSR(연율화)(배)   1.24
165703  A005930         삼성전자 2025-06-30  수정PSR(연율화)(배)   1.35
